# dbt 2: Tests and contracts

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/dbt/dbt-02-tests-and-contracts.ipynb)

Run dbt locally against DuckDB using a fresh project for each lesson.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q dbt-core~=1.9 dbt-duckdb~=1.9 duckdb~=2.0

## Setup and sample data

Create a small dbt project and database in a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import duckdb
from dbt.cli.main import dbtRunner

workspace = TemporaryDirectory()
project = Path(workspace.name)
models = project / "models"
models.mkdir()
warehouse = project / "warehouse.duckdb"
(project / "dbt_project.yml").write_text("name: handbook\nversion: '1.0'\nprofile: handbook\n")
(project / "profiles.yml").write_text(
    f"handbook:\n  outputs:\n    dev:\n      type: duckdb\n      path: '{warehouse}'\n      threads: 1\n"
    "  target: dev\n"
)


def dbt(*args, expect_success=True):
    result = dbtRunner().invoke([*args, "--project-dir", str(project), "--profiles-dir", str(project)])
    if result.success != expect_success:
        raise RuntimeError(f"Unexpected dbt result: {result.exception or result.result}")
    print("dbt completed:", result.success)


def query(sql):
    with duckdb.connect(str(warehouse)) as connection:
        print(connection.execute(sql).fetchall())

## Tests and contracts

A contract checks the published columns and types. A data test queries built data for violations of a rule.

**Where it matters**

Use both schema and data checks before promoting a model.

**Watch out for**

A model can have the correct schema and still contain duplicate business keys.

In [ ]:
(models / "orders.sql").write_text("select * from (values (1, 1200), (2, 800)) t(id, cents)")
(models / "schema.yml").write_text("""
version: 2
models:
  - name: orders
    config:
      materialized: table
      contract:
        enforced: true
    columns:
      - name: id
        data_type: integer
        data_tests: [not_null, unique]
      - name: cents
        data_type: integer
""")
dbt("build")
(models / "orders.sql").write_text("select * from (values (1, 1200), (1, 800)) t(id, cents)")
dbt("build", expect_success=False)

**What to notice:** The duplicate fixture still satisfies the integer contract. The uniqueness test reports the duplicate key and dbt returns failure.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Which test catches an unchanged integer key appearing twice?

In [ ]:
# Try your solution here.

In [ ]:
print("The unique data test.")

## Key takeaways

- Use both schema and data checks before promoting a model.

**Next:** [dbt 3](dbt-03-incremental-models.ipynb)